# QosTil · дообучение Whisper Small

Научим модель записывать казахские и русские слова в том порядке, в котором они прозвучали. Пройдём путь от таблицы с разметкой до оценки WER и сохранения весов.

Открой этот `.ipynb` через **File → Upload notebook** в [Google Colab](https://colab.research.google.com/). Затем выбери **Runtime → Change runtime type → GPU** и запускай ячейки сверху вниз. [Справка Colab по GPU](https://colab.research.google.com/notebooks/pro.ipynb)

## 1. Собери и разметь записи

1. Для первого запуска собери примерно 50–100 коротких клипов от нескольких людей, согласных на использование записей и их загрузку в Colab. Пусть люди говорят своими словами. Оставляй одного говорящего на клип. При разрезании длинной записи сохраняй смену языка внутри клипа. Подойдут `.m4a`, `.wav` и `.mp3`. Для этой тетради длительность каждого клипа должна быть до 30 секунд. Удобно начать с 5–15 секунд.
2. Прослушай клип и запиши дословную расшифровку, сохраняя казахские буквы `ә`, `ғ`, `қ`, `ң`, `ө`, `ұ`, `ү`, `һ`, `і` и исходный язык каждого слова. Если Whisper сделал черновик, исправь его по записи. Сверяй расшифровки теста вручную.
3. Назначь каждому человеку один анонимный `speaker_id`. Заранее выбери, чьи записи пойдут в `train` для обучения, чьи в `validation` для выбора лучшей эпохи и чьи в `test` для итоговой оценки. В каждой части нужны смешанные фразы. Оставь несколько голосов для проверки, если позволяет объём данных. Правило разбиения по людям действует и для разных клипов из одной беседы.

В папке `dataset` создай папку `audio` и текстовый файл `metadata.csv` в UTF-8. Вот пример трёх строк с условными именами файлов и людей. Подставь реальные данные. Если в тексте есть запятая, оберни поле в двойные кавычки.

```csv
audio_path,text,speaker_id,split
audio/001.m4a,"Бүгін сабақтан кейін созвонимся, хорошо?",spk_01,train
audio/002.m4a,"Я уже в автобусе, бес минутта келемін.",spk_02,validation
audio/003.m4a,"Сен бара бер, потом я догоню.",spk_03,test
```

В QosTil можно взять за образец `sample-records/` и `references.json`. Там уже есть пары «аудио и текст». Чтобы использовать эти записи в эксперименте, добавь проверенный `speaker_id` для каждой. Четыре клипа подходят для проверки чтения аудио. Для сравнения качества собери больше записей с разными голосами.

Из папки `dataset` выполни в терминале macOS команду ниже. В ZIP на верхнем уровне должны лежать `metadata.csv` и `audio/`.

```bash
zip -r qostil-dataset.zip metadata.csv audio
```

## 2. Установи зависимости и загрузи ZIP

PyAV прочитает `.m4a` и преобразует звук. Transformers и PyTorch выполнят обучение. Для пробного запуска с несколькими обновлениями весов поменяй `MAX_STEPS` на `5`. Значение `-1` запускает все эпохи из `NUM_EPOCHS`.

In [ ]:
%pip install -q "transformers==4.57.1" "accelerate==1.11.0" "av==18.1.1" "jiwer==4.0.0"

In [ ]:
from os import environ
from pathlib import Path
from tempfile import mkdtemp

import torch

MODEL_ID = "openai/whisper-small"
LANGUAGE = "kk"
NUM_EPOCHS = 3
GRAD_ACCUMULATION = 4
MAX_STEPS = -1
GENERATION_MAX_LENGTH = 225
USE_CPU = False

# Сбрасываем смешанную точность, оставшуюся после предыдущего запуска Trainer.
environ["ACCELERATE_MIXED_PRECISION"] = "no"

if not USE_CPU and not torch.cuda.is_available():
    raise RuntimeError("Подключи GPU в Colab через Runtime → Change runtime type и запусти ячейку снова.")
OUTPUT_DIR = Path(mkdtemp(prefix="qostil-whisper-", dir="/content"))
print("Устройство:", "CPU" if USE_CPU else torch.cuda.get_device_name(0))

In [ ]:
from tempfile import mkdtemp
from zipfile import ZipFile

from google.colab import files

uploaded = files.upload()  # Выбери qostil-dataset.zip.
if len(uploaded) != 1:
    raise ValueError("Загрузи один ZIP с metadata.csv и папкой audio.")
archive_path = next(iter(uploaded))
with ZipFile(archive_path) as archive:
    data_root = Path(mkdtemp(prefix="qostil-dataset-"))
    archive.extractall(data_root)
del uploaded
print("Датасет распакован в", data_root)

## 3. Проверь разметку и подготовь признаки

Сначала проверяем, что все файлы существуют и голоса из теста не попали в обучение. Затем переводим аудио в моно 16 кГц и превращаем его в спектрограммы, а текст — в токены Whisper. Здесь используем казахскую языковую подсказку `kk` для преимущественно казахских фраз и задачу `transcribe`. Код языка в начале последовательности служит подсказкой, а текстовый эталон сохраняет слова на обоих языках. Оставляем одинаковую подсказку при сравнении двух моделей. [Руководство Transformers по Whisper](https://huggingface.co/blog/fine-tune-whisper)

In [ ]:
import csv
from pathlib import Path

import av
import jiwer
import numpy as np

SAMPLE_RATE = 16_000
MAX_SECONDS = 30
SPLIT_NAMES = ("train", "validation", "test")
NORMALIZE = jiwer.Compose([
    jiwer.ToLowerCase(),
    jiwer.RemovePunctuation(),
    jiwer.RemoveMultipleSpaces(),
    jiwer.Strip(),
])


def load_manifest(root):
    root = root.resolve()
    manifest = root / "metadata.csv"
    if not manifest.is_file():
        raise ValueError("В корне ZIP должен быть metadata.csv. Проверь, не вложена ли папка dataset целиком.")
    splits = {name: [] for name in SPLIT_NAMES}
    speaker_splits = {}
    seen_paths = set()
    expected_columns = {"audio_path", "text", "speaker_id", "split"}
    with manifest.open(encoding="utf-8-sig", newline="") as file:
        reader = csv.DictReader(file)
        if not expected_columns.issubset(reader.fieldnames or []):
            raise ValueError("Нужны столбцы audio_path,text,speaker_id,split. Разделитель — запятая.")
        for line, row in enumerate(reader, start=2):
            if None in row or any(not (row.get(key) or "").strip() for key in expected_columns):
                raise ValueError(f"Строка {line}: проверь поля и кавычки вокруг текста с запятыми.")
            audio_path = (root / row["audio_path"].strip()).resolve()
            text = row["text"].strip()
            speaker = row["speaker_id"].strip()
            split = row["split"].strip()
            if split not in splits:
                raise ValueError(f"Строка {line}: неизвестная часть датасета '{split}'.")
            if not audio_path.is_relative_to(root / "audio") or not audio_path.is_file():
                raise ValueError(f"Строка {line}: файла {row['audio_path']} нет в папке audio.")
            if audio_path in seen_paths or not NORMALIZE(text):
                raise ValueError(f"Строка {line}: файл повторяется или текст пуст.")
            if speaker_splits.setdefault(speaker, split) != split:
                raise ValueError(f"Строка {line}: голос '{speaker}' попал в несколько частей.")
            seen_paths.add(audio_path)
            splits[split].append({"audio_path": audio_path, "text": text, "speaker_id": speaker})
    for split, rows in splits.items():
        if not rows:
            raise ValueError(f"Добавь хотя бы одну запись в '{split}'.")
    return splits


def read_audio(path):
    resampler = av.AudioResampler(format="fltp", layout="mono", rate=SAMPLE_RATE)
    chunks = []
    with av.open(str(path)) as container:
        if not container.streams.audio:
            raise ValueError(f"{path.name}: не найден аудиопоток.")
        for frame in container.decode(audio=0):
            chunks.extend(output.to_ndarray().reshape(-1) for output in resampler.resample(frame))
        chunks.extend(output.to_ndarray().reshape(-1) for output in resampler.resample(None))
    sample_count = sum(chunk.size for chunk in chunks)
    if not 0 < sample_count <= SAMPLE_RATE * MAX_SECONDS:
        raise ValueError(f"{path.name}: нужен непустой клип длительностью до {MAX_SECONDS} секунд.")
    return np.concatenate(chunks)

In [ ]:
splits = load_manifest(data_root)
for split, rows in splits.items():
    speakers = {row["speaker_id"] for row in rows}
    print(f"{split}: {len(rows)} клипов, {len(speakers)} голосов")

In [ ]:
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained(MODEL_ID, language=LANGUAGE, task="transcribe")


def prepare_example(row):
    samples = read_audio(row["audio_path"])
    features = processor.feature_extractor(
        samples,
        sampling_rate=SAMPLE_RATE,
        return_attention_mask=True,
        return_tensors="np",
    )
    labels = processor.tokenizer(row["text"]).input_ids
    # Оставляем место для конца последовательности при генерации.
    if len(labels) >= GENERATION_MAX_LENGTH:
        raise ValueError(f"{row['audio_path'].name}: раздели длинную фразу на более короткие клипы.")
    return {
        "input_features": features.input_features[0],
        "attention_mask": features.attention_mask[0],
        "labels": labels,
    }


datasets = {split: [prepare_example(row) for row in rows] for split, rows in splits.items()}
print("Форма спектрограммы первого клипа:", datasets["train"][0]["input_features"].shape)

## 4. Запусти дообучение

Энкодер извлекает признаки из аудио, декодер выводит текст. В первом запуске фиксируем веса энкодера и обучаем только декодер, чтобы снизить требования к памяти. Обучение использует FP32: в пробном запуске на Tesla T4 режим FP16 дал `NaN` в градиентах и пропустил обновление весов. За одну эпоху модель проходит весь `train`. `eval_wer` в таблице обучения соответствует `validation`. Меньше WER — лучше. По этой метрике Trainer выбирает веса лучшей эпохи. Для короткой проверки с `MAX_STEPS = 5` оценки качества будут ориентировочными. Если повторяешь эксперимент, начни с ячейки настроек, чтобы заново загрузить исходную модель и сохранить результаты в новую папку.

In [ ]:
import torch
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
model.freeze_encoder()
model.generation_config.language = LANGUAGE
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None


def collate(examples):
    # Спектрограммы уже одной длины. Тексты дополняем до общей длины внутри батча.
    tokens = processor.tokenizer.pad(
        [{"input_ids": example["labels"]} for example in examples],
        return_tensors="pt",
    )
    labels = tokens.input_ids.masked_fill(tokens.attention_mask == 0, -100)
    # Декодер сам добавляет стартовый токен, поэтому снимаем его с эталона.
    labels = labels[:, 1:]
    return {
        "input_features": torch.from_numpy(np.stack([example["input_features"] for example in examples])),
        "attention_mask": torch.from_numpy(np.stack([example["attention_mask"] for example in examples])),
        "labels": labels,
    }


def decode(token_ids):
    token_ids = np.where(token_ids == -100, processor.tokenizer.pad_token_id, token_ids)
    return processor.tokenizer.batch_decode(token_ids, skip_special_tokens=True)


def compute_metrics(result):
    predictions = [NORMALIZE(text) for text in decode(result.predictions)]
    references = [NORMALIZE(text) for text in decode(result.label_ids)]
    return {"wer": 100 * jiwer.wer(references, predictions), "cer": 100 * jiwer.cer(references, predictions)}


args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    use_cpu=USE_CPU,
    fp16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUMULATION,
    learning_rate=1e-5,
    optim="adamw_torch",
    num_train_epochs=NUM_EPOCHS,
    max_steps=MAX_STEPS,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    save_only_model=True,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=GENERATION_MAX_LENGTH,
    generation_num_beams=1,
    logging_steps=5,
    logging_first_step=True,
    report_to="none",
    push_to_hub=False,
    seed=42,
)
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=datasets["train"],
    eval_dataset=datasets["validation"],
    data_collator=collate,
    processing_class=processor,
    compute_metrics=compute_metrics,
)

In [ ]:
# Сохраняем предсказания базового Whisper для сравнения после обучения.
baseline = trainer.predict(datasets["test"], metric_key_prefix="base")
trainer.train()

model_dir = OUTPUT_DIR / "model"
trainer.save_model(str(model_dir))
print("Модель и процессор сохранены в", model_dir)

## 5. Сравни результат и скачай веса

Сравни WER (ошибки в словах) и CER (ошибки в символах) на одних и тех же фразах из `test`. Строки под таблицей покажут, улучшилось ли распознавание около смены языка. На маленьком тесте даже одно слово может сильно изменить WER.

In [ ]:
adapted = trainer.predict(datasets["test"], metric_key_prefix="tuned")
for name, result, prefix in [("До", baseline, "base"), ("После", adapted, "tuned")]:
    print(f"{name}: WER {result.metrics[prefix + '_wer']:.1f}% · CER {result.metrics[prefix + '_cer']:.1f}%")

examples = zip(splits["test"][:5], decode(baseline.predictions[:5]), decode(adapted.predictions[:5]))
for row, before, after in examples:
    print(f"\nЭталон: {row['text']}\nДо:     {before}\nПосле:  {after}")

In [ ]:
from shutil import make_archive

model_archive = make_archive(str(OUTPUT_DIR / "qostil-whisper-small"), "zip", root_dir=model_dir)
files.download(model_archive)

В архиве будут веса и файлы процессора. Скачай их до отключения среды Colab. После распаковки подставь путь к папке `model_dir`. Языковая подсказка и задача сохранены в конфигурации генерации. Передай их токенизатору явно, если начнёшь новый запуск обучения с этих весов.

```python
restored = WhisperForConditionalGeneration.from_pretrained(model_dir)
restored_processor = WhisperProcessor.from_pretrained(
    model_dir,
    language=restored.generation_config.language,
    task=restored.generation_config.task,
)
```

Чтобы подключить модель к `faster-whisper` в QosTil, понадобится преобразовать её в CTranslate2 и передать путь к преобразованной модели. [Документация faster-whisper](https://github.com/SYSTRAN/faster-whisper#model-conversion)